In [ ]:
from sentence_transformers import SentenceTransformer
import tqdm as notebook_tqdm
import torch
from tqdm import tqdm
import os

# ============================================
# Load Qwen3-VL-Embedding-2B GGUF Model
# ============================================
# Model: https://huggingface.co/DevQuasar/Qwen.Qwen3-VL-Embedding-2B-GGUF
# This is a Q4 quantized version for efficient inference

MODEL_ID = "DevQuasar/Qwen.Qwen3-VL-Embedding-2B-GGUF"
GGUF_FILENAME = "Qwen.Qwen3-VL-Embedding-2B.Q4_K_M.gguf"  # Common Q4 quantization filename

# Try loading from HuggingFace Hub first (sentence-transformers may handle GGUF)
try:
    print(f"Loading model from Hub: {MODEL_ID}")
    model = SentenceTransformer(MODEL_ID, model_kwargs={"torch_dtype": "float16"})
    print("Model loaded successfully!")
except Exception as e:
    print(f"Direct Hub loading failed: {e}")
    print("Attempting GGUF-specific loading...")
    
    # Fallback: Download GGUF file and load with llama-cpp-python
    from huggingface_hub import hf_hub_download
    
    # Download the GGUF file
    print(f"Downloading {GGUF_FILENAME} from Hub...")
    gguf_path = hf_hub_download(
        repo_id=MODEL_ID,
        filename=GGUF_FILENAME,
        cache_dir="./models"
    )
    print(f"GGUF file downloaded to: {gguf_path}")
    
    # Try loading with llama-cpp-python for GGUF support
    try:
        from llama_cpp import Llama
        
        class GGUFEmbeddingModel:
            """
            Wrapper for llama-cpp-python GGUF model to provide sentence-transformers-like API.
            """
            def __init__(self, model_path, embedding=True):
                self.llm = Llama(
                    model_path=model_path,
                    n_ctx=2048,
                    n_threads=8,
                    embedding=True,  # Enable embedding mode
                    verbose=False
                )
                self.embedding_dim = self.llm.n_embd()
                print(f"GGUF model loaded. Embedding dimension: {self.embedding_dim}")
            
            def encode(self, texts, prompt_name=None, **kwargs):
                """Encode texts into embeddings."""
                if isinstance(texts, str):
                    texts = [texts]
                
                # Apply query prompt if specified
                if prompt_name == "query":
                    # Some Qwen models use a prefix for queries
                    texts = [f"[Q] {text}" for text in texts]
                
                embeddings = []
                for text in texts:
                    emb = self.llm.embed(text)
                    embeddings.append(emb)
                
                import numpy as np
                return np.array(embeddings)
            
            def similarity(self, vec1, vec2):
                """Compute cosine similarity between vectors."""
                import numpy as np
                vec1 = vec1 / (np.linalg.norm(vec1, axis=-1, keepdims=True) + 1e-8)
                vec2 = vec2 / (np.linalg.norm(vec2, axis=-1, keepdims=True) + 1e-8)
                return torch.tensor(vec1 @ vec2.transpose(-1, -2))
        
        model = GGUFEmbeddingModel(gguf_path)
        print("GGUF model loaded successfully with llama-cpp-python!")
    except ImportError:
        print("llama-cpp-python not installed. Install with: pip install llama-cpp-python")
        raise
    except Exception as e:
        print(f"GGUF loading failed: {e}")
        raise

/home/tom/bre-fine-arts/Fine-Arts-Main/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model from Hub: DevQuasar/Qwen.Qwen3-VL-Embedding-2B-GGUF
Direct Hub loading failed: Unrecognized model in DevQuasar/Qwen.Qwen3-VL-Embedding-2B-GGUF. Should have a `model_type` key in its config.json.
Attempting GGUF-specific loading...


RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-69ee7248-51a475024ae71dd25bafd3b4;866f0933-4525-4d75-a98e-4d2d0ce2d242)

Entry Not Found for url: https://huggingface.co/DevQuasar/Qwen.Qwen3-VL-Embedding-2B-GGUF/resolve/main/model-q4_k.gguf.

In [ ]:
# Import required libraries for image downloading and display
import requests
from requests.auth import HTTPBasicAuth
from io import BytesIO
from PIL import Image as PILImage
from IPython.display import display
from dotenv import load_dotenv
load_dotenv()

# Nextcloud configuration
NC_HOST = os.getenv("DB_HOST")
NC_ACC = os.getenv("NC_ACC")
NC_PASS = os.getenv("NC_PASS")

print(f"Nextcloud host: {NC_HOST}")
print(f"Nextcloud account: {NC_ACC}")

In [ ]:
def get_images(file_id, file_path):
    """
    Download and display a thumbnail/preview for a file from Nextcloud.
    
    Args:
        file_id: The Nextcloud file ID
        file_path: The path to the file on the Nextcloud server
    
    Returns:
        PIL.Image: The resized 256x256 PIL Image
    """
    server_url = f'http://{NC_HOST}:8080/remote.php/dav/files/{NC_ACC}'
    preview_url = f'http://{NC_HOST}:8080/core/preview?fileId={file_id}&x=1080&y=1080'
    username = NC_ACC
    password = NC_PASS

    file_url = f"{server_url}{file_path}"

    response = requests.get(preview_url, auth=HTTPBasicAuth(username, password), stream=True)
 
    if response.status_code == 200:
        file_in_memory = BytesIO()
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                file_in_memory.write(chunk)
        file_in_memory.seek(0)
        img = PILImage.open(file_in_memory)
        img = img.resize((256, 256))
        display(img)
    elif response.status_code == 404:
        try:
            response = requests.get(file_url, auth=HTTPBasicAuth(username, password), stream=True)
            file_in_memory = BytesIO()
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    file_in_memory.write(chunk)
            file_in_memory.seek(0)
            source_img = PILImage.open(file_in_memory)
            img_display = source_img.resize((256, 256))
            display(img_display)
        except Exception as e:
            print(f"Failed to download file. Exception: {e}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")
    
    try:
        import gc
        del file_in_memory
        gc.collect()
    except:
        pass
    return img

In [ ]:
from sqlalchemy import create_engine, MetaData, Table, select, insert
from sqlalchemy.exc import SQLAlchemyError
import pandas as pd

def create_db_connection():
    load_dotenv()
    DB_HOST = os.getenv("DB_HOST")
    DB_NAME = os.getenv("DB_NAME")
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    engine = create_engine('postgresql+pg8000://'+DB_USER+':'+DB_PASSWORD+'@'+DB_HOST+':5432/'+DB_NAME)
    return engine

def get_tags():
    try:
        engine = create_db_connection()
        df_systag = pd.read_sql_table('oc_systemtag', engine)
        df_tagmap = pd.read_sql_table('oc_systemtag_object_mapping', engine)
    except SQLAlchemyError as e:
        print(f"An error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    return df_systag, df_tagmap

In [ ]:
print("Loading tags from database...")
df_systag, df_tagmap = get_tags()
print(f"Loaded {len(df_systag)} tags and {len(df_tagmap)} mappings")

In [ ]:
# ============================================
# Document Aggregation Approach
# ============================================
# Build a document for each file by aggregating all its tags.
# This creates a richer semantic representation per file.

def build_file_documents(df_systag, df_tagmap):
    """
    Build a dictionary mapping file IDs to their aggregated tag documents.
    
    Args:
        df_systag: DataFrame with tag id and name columns
        df_tagmap: DataFrame with systemtagid and objectid columns
    
    Returns:
        dict: {file_id: "tag1 tag2 tag3 ..."}
    """
    # Merge tagmap with tag names
    tagmap_with_names = df_tagmap.merge(
        df_systag[['id', 'name']],
        left_on='systemtagid',
        right_on='id',
        how='inner'
    )
    
    # Group by file_id (objectid) and aggregate tag names
    file_docs = {}
    for file_id, group in tagmap_with_names.groupby('objectid'):
        # Sort tags alphabetically for consistency, deduplicate
        tags = sorted(set(group['name'].tolist()))
        file_docs[str(file_id)] = ' '.join(tags)
    
    return file_docs, tagmap_with_names

file_documents, tagmap_with_names = build_file_documents(df_systag, df_tagmap)
print(f"Built {len(file_documents)} file documents")

# Show some examples
print("\nExample file documents:")
for i, (file_id, doc) in enumerate(list(file_documents.items())[:5]):
    print(f"  File {file_id}: '{doc}'")

In [ ]:
# ============================================
# TF-IDF Weighted Document Embeddings
# ============================================
# Apply TF-IDF weighting to tags so rare/discriminative tags
# contribute more to the embedding than common tags.

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("Applying TF-IDF weighting to documents...")

# Get document texts in consistent order
doc_ids = list(file_documents.keys())
doc_texts = [file_documents[fid] for fid in doc_ids]

# Fit TF-IDF vectorizer on all documents
tfidf_vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
tfidf_matrix = tfidf_vectorizer.fit_transform(doc_texts)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

# Convert TF-IDF weighted sparse matrix back to dense text
# by repeating tags proportional to their TF-IDF scores
def tfidf_weighted_docs(tfidf_matrix, doc_ids, doc_texts, repeat_factor=3):
    """
    Create weighted document strings by repeating tags based on TF-IDF scores.
    
    Args:
        tfidf_matrix: Sparse TF-IDF matrix (n_docs x n_terms)
        doc_ids: List of document IDs
        doc_texts: List of original document texts
        repeat_factor: Maximum repeats for high-scoring terms
    
    Returns:
        list: Weighted document strings
    """
    vocabulary = tfidf_vectorizer.vocabulary_
    inv_vocab = {v: k for k, v in vocabulary.items()}
    
    weighted_docs = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix.getrow(i)
        indices = row.nonzero()[1]
        scores = row.data
        
        # Sort by TF-IDF score descending
        sorted_order = np.argsort(scores)[::-1]
        indices = indices[sorted_order]
        scores = scores[sorted_order]
        
        # Repeat tags based on score (higher score = more repeats)
        weighted_tags = []
        for idx, score in zip(indices, scores):
            tag = inv_vocab[idx]
            # Scale repeats: min 1, max repeat_factor based on score
            repeats = max(1, int(score * repeat_factor))
            weighted_tags.extend([tag] * repeats)
        
        weighted_docs.append(' '.join(weighted_tags))
    
    return weighted_docs

# Create weighted documents (repeat_factor=3 means top tags repeat ~3x)
weighted_doc_texts = tfidf_weighted_docs(tfidf_matrix, doc_ids, doc_texts, repeat_factor=3)

# Show comparison
print("\nOriginal vs TF-IDF Weighted Documents:")
for i in range(min(5, len(doc_ids))):
    print(f"\n  File {doc_ids[i]}:")
    print(f"    Original:   '{doc_texts[i][:100]}...'")
    print(f"    Weighted:   '{weighted_doc_texts[i][:100]}...'")

In [ ]:
# ============================================
# Pre-compute TF-IDF Weighted Document Embeddings
# ============================================

print("Encoding TF-IDF weighted documents...")
tfidf_doc_embeddings = model.encode(weighted_doc_texts)
print(f"Encoded {len(tfidf_doc_embeddings)} weighted documents")
print(f"Embedding shape: {tfidf_doc_embeddings.shape}")

In [ ]:
# ============================================
# TF-IDF Weighted Document Similarity Search
# ============================================

def search_files_by_tfidf_document_similarity(query, top_k=10):
    """
    Search for files using TF-IDF weighted document embeddings.
    
    Args:
        query: Natural language search query
        top_k: Number of top results to return
    
    Returns:
        DataFrame with file_id, document, similarity score, ranked
    """
    # Encode the query
    query_embedding = model.encode(query, prompt_name="query")
    
    # Compute cosine similarity between query and all weighted document embeddings
    similarity = model.similarity(
        torch.tensor(query_embedding).unsqueeze(0),
        torch.tensor(tfidf_doc_embeddings)
    )
    
    # Get top-k indices
    top_indices = similarity[0].argsort(descending=True)[:top_k]
    
    # Build results DataFrame
    results = pd.DataFrame({
        'file_id': [doc_ids[i] for i in top_indices],
        'document': [doc_texts[i] for i in top_indices],
        'weighted_document': [weighted_doc_texts[i] for i in top_indices],
        'similarity': similarity[0][top_indices].cpu().numpy()
    })
    
    return results

print("TF-IDF weighted search function ready!")

In [ ]:
# ============================================
# Display Thumbnails for Search Results
# ============================================

def display_doc_search_thumbnails(results, top_n=3):
    """
    Display thumbnails for the top N files from document search results.
    
    Args:
        results: DataFrame with file_id column
        top_n: Number of thumbnails to display (default 3)
    """
    if results.empty:
        print("No files found.")
        return
    
    print(f"\nDisplaying top {min(top_n, len(results))} thumbnails:")
    for i, row in results.head(top_n).iterrows():
        file_id = row['file_id']
        print(f"Thumbnail {i+1}: File ID = {file_id}, Similarity = {row['similarity']:.4f}")
        file_path = "/images/"
        try:
            img = get_images(file_id, file_path)
        except Exception as e:
            print(f"  Failed to load image: {e}")
        print()

In [ ]:
# ============================================
# Run TF-IDF Weighted Document Search Tests
# ============================================

test_queries = [
    "A painting with green landscape scene",
    "portrait of a person",
    "abstract art with bright colors",
]

for query in tqdm(test_queries):
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    # Search using TF-IDF weighted document similarity
    results = search_files_by_tfidf_document_similarity(query, top_k=10)
    
    print(f"\nTop 10 results:")
    print(results[['file_id', 'document', 'similarity']].to_string(index=False))
    
    # Display top 3 thumbnails
    display_doc_search_thumbnails(results, top_n=3)

In [ ]:
# ============================================
# Custom Test Queries
# ============================================

custom_queries = [
    "coloful contemproary art for a wallpaper",
    "greek villages",
    "primarily green wallpaper patterns which are repetitive with flowers",
    "show me lemmons!",
]

for query in tqdm(custom_queries):
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    # Search using TF-IDF weighted document similarity
    results = search_files_by_tfidf_document_similarity(query, top_k=10)
    
    print(f"\nTop 10 results:")
    print(results[['file_id', 'document', 'similarity']].to_string(index=False))
    
    # Display top 5 thumbnails
    display_doc_search_thumbnails(results, top_n=5)